PART 1


1. Baseline Model:

In [3]:
# Setup and Dataset Preparation
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss

# 1. Set seed for complete reproducibility
RANDOM_STATE = 42

# 2. Create a balanced binary classification dataset (2000 samples, 20 features)
X, y = make_classification(
    n_samples=2000,
    n_features=20,
    n_informative=15,
    n_redundant=3,
    n_classes=2,
    random_state=RANDOM_STATE
)

# 3. Split dataset into 80% training set and 20% validation set
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

# 4. List to store results for the final summary table (Part 5)
results_summary = []

print(f"Data ready: {X_train.shape[0]} train rows, {X_val.shape[0]} validation rows.")

Data ready: 1600 train rows, 400 validation rows.


In [5]:
# Part 1.1: Baseline Single Unpruned Decision Tree (max_depth=None)
from sklearn.tree import DecisionTreeClassifier

# 1. Measure execution runtime
start_time = time.time()

# 2. Fit a single, unpruned Decision Tree (max_depth=None)
baseline_tree = DecisionTreeClassifier(max_depth=None, random_state=RANDOM_STATE)
baseline_tree.fit(X_train, y_train)

dt_time = time.time() - start_time

# 3. Compute training and validation accuracies
train_acc_dt = baseline_tree.score(X_train, y_train)
val_acc_dt = baseline_tree.score(X_val, y_val)

# 4. Compute error rates (Error = 1.0 - Accuracy)
train_err_dt = 1.0 - train_acc_dt
val_err_dt = 1.0 - val_acc_dt

# 5. Append to summary table records
results_summary.append({
    "Model": "Baseline (Unpruned Tree)",
    "Train Accuracy": round(train_acc_dt, 4),
    "Val Accuracy": round(val_acc_dt, 4),
    "OOB Error": "N/A",
    "Train Time (s)": round(dt_time, 4)
})

# 6. Display metrics
print("Part 1: Baseline Decision Tree")
print(f"Training Accuracy:   {train_acc_dt:.4f}  | Training Error:   {train_err_dt:.4f}")
print(f"Validation Accuracy: {val_acc_dt:.4f}  | Validation Error: {val_err_dt:.4f}")
print(f"Execution Time:      {dt_time:.4f} seconds")

Part 1: Baseline Decision Tree
Training Accuracy:   1.0000  | Training Error:   0.0000
Validation Accuracy: 0.8325  | Validation Error: 0.1675
Execution Time:      0.0216 seconds


2. Bagging & Random Forest:

In [7]:
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier

# 2.1 Bagged Decision Tree Ensemble with 100 base estimators
start_time = time.time()

bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=None, random_state=RANDOM_STATE),
    n_estimators=100,
    oob_score=True,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
bagging_model.fit(X_train, y_train)
bag_time = time.time() - start_time

bag_train_acc = bagging_model.score(X_train, y_train)
bag_val_acc = bagging_model.score(X_val, y_val)
bag_oob_err = 1.0 - bagging_model.oob_score_
bag_val_err = 1.0 - bag_val_acc

results_summary.append({
    "Model": "Bagging (100 Trees)",
    "Train Accuracy": round(bag_train_acc, 4),
    "Val Accuracy": round(bag_val_acc, 4),
    "OOB Error": round(bag_oob_err, 4),
    "Train Time (s)": round(bag_time, 4)
})

# 2.2 Random Forest Classifier with 100 base estimators
start_time = time.time()

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_features="sqrt",
    oob_score=True,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_time = time.time() - start_time

rf_train_acc = rf_model.score(X_train, y_train)
rf_val_acc = rf_model.score(X_val, y_val)
rf_oob_err = 1.0 - rf_model.oob_score_
rf_val_err = 1.0 - rf_val_acc

results_summary.append({
    "Model": "Random Forest (100 Trees)",
    "Train Accuracy": round(rf_train_acc, 4),
    "Val Accuracy": round(rf_val_acc, 4),
    "OOB Error": round(rf_oob_err, 4),
    "Train Time (s)": round(rf_time, 4)
})

# 2.3 Output comparison between OOB Error and Validation Error
print("Part 2: OOB Error vs. Validation Error")
print(f"Bagging       -> OOB Error: {bag_oob_err:.4f} | Validation Error: {bag_val_err:.4f}")
print(f"Random Forest -> OOB Error: {rf_oob_err:.4f} | Validation Error: {rf_val_err:.4f}")

Part 2: OOB Error vs. Validation Error
Bagging       -> OOB Error: 0.1031 | Validation Error: 0.0900
Random Forest -> OOB Error: 0.1000 | Validation Error: 0.0900
